In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from geobr import read_municipality
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('dados', exist_ok=True)

print("Baixando dados dos municípios de MG...")
mg = read_municipality(code_muni="MG", year=2020)
mg = mg.to_crs(epsg=31983)
mg['area_km2'] = mg.geometry.area / (10**6)
mg.to_file('dados/municipios-mg.geojson', driver='GeoJSON')
print("✅ 'municipios-mg.geojson' salvo!")

print("Gerando dados socioeconômicos simulados...")
nomes_muni = mg['name_muni'].unique()
df_ibge = pd.DataFrame({
    'municipio': nomes_muni,
    'populacao_2022': np.random.randint(5000, 2500000, len(nomes_muni)),
    'pib_mil_reais': np.random.randint(10000, 90000000, len(nomes_muni))
})
df_ibge.to_csv('dados/populacao-pib-municipios-mg.csv', index=False)
print("✅ 'populacao-pib-municipios-mg.csv' salvo!")

print("Unificando focos de desmatamento...")
# Aqui ele pega os arquivos que você gerou na Célula 2
ago = gpd.read_file('dados/desmatamento_ago22.gpkg')
setem = gpd.read_file('dados/desmatamento_set_22.gpkg')
ago['mes'] = 'Agosto'
setem['mes'] = 'Setembro'

focos = pd.concat([ago, setem], ignore_index=True)
focos = gpd.GeoDataFrame(focos, geometry='geometry', crs="EPSG:4326")
focos = focos.to_crs(epsg=31983)
focos.to_file('dados/focos-desmatamento-mg.geojson', driver='GeoJSON')
print("✅ 'focos-desmatamento-mg.geojson' salvo com sucesso!")

In [ ]:
import geopandas as gpd
import pandas as pd
from IPython.display import display

# Carregar os dados que foram gerados na Célula 3
municipios = gpd.read_file('dados/municipios-mg.geojson')
focos = gpd.read_file('dados/focos-desmatamento-mg.geojson')
socioeco = pd.read_csv('dados/populacao-pib-municipios-mg.csv')

# Calcular áreas
focos['area_desmatada_ha'] = focos.geometry.area / 10000
focos['area_desmatada_km2'] = focos.geometry.area / 10**6

# Cruzamento Espacial (quais focos caem em quais municípios)
focos_muni = gpd.sjoin(focos, municipios, how="inner", predicate="intersects")

print("--- 1. Área total desmatada por mês (Hectares) ---")
area_mes_ha = focos.groupby('mes')['area_desmatada_ha'].sum().reset_index()
display(area_mes_ha)

print("\n--- 2. Área total desmatada por Bioma (km²) ---")
area_bioma_km2 = focos.groupby('bioma')['area_desmatada_km2'].sum().reset_index()
display(area_bioma_km2)

print("\n--- 3. Área total desmatada por Município e Mês (km²) [Top 10] ---")
area_muni_mes_km2 = focos_muni.groupby(['name_muni', 'mes'])['area_desmatada_km2'].sum().reset_index()
display(area_muni_mes_km2.sort_values(by='area_desmatada_km2', ascending=False).head(10))

In [ ]:
import geopandas as gpd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

# Carregar os dados
municipios = gpd.read_file('dados/municipios-mg.geojson')
focos = gpd.read_file('dados/focos-desmatamento-mg.geojson')
socioeco = pd.read_csv('dados/populacao-pib-municipios-mg.csv') # Tem que ser o arquivo real!

# Calcular áreas
focos['area_desmatada_ha'] = focos.geometry.area / 10000
focos['area_desmatada_km2'] = focos.geometry.area / 10**6

# Cruzamento Espacial
focos_muni = gpd.sjoin(focos, municipios, how="inner", predicate="intersects")

print("1. Área total desmatada por mês (Hectares) ")
area_mes_ha = focos.groupby('mes')['area_desmatada_ha'].sum().reset_index()
display(area_mes_ha)

print("\n 2. Área total desmatada por Bioma (km²)")
area_bioma_km2 = focos.groupby('bioma')['area_desmatada_km2'].sum().reset_index()
display(area_bioma_km2)

print("\n 3. Área total desmatada por Município e Mês (km²) [Top 10] ")
area_muni_mes_km2 = focos_muni.groupby(['name_muni', 'mes'])['area_desmatada_km2'].sum().reset_index()
display(area_muni_mes_km2.sort_values(by='area_desmatada_km2', ascending=False).head(10))

print("\n4. Matriz de Correlação ")
desmatamento_total_muni = focos_muni.groupby('name_muni')['area_desmatada_ha'].sum().reset_index()
df_correlacao = pd.merge(desmatamento_total_muni, socioeco, left_on='name_muni', right_on='municipio', how='inner')
correlacao = df_correlacao[['area_desmatada_ha', 'populacao_2022', 'pib_mil_reais']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlacao, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title('Matriz de Correlação: Desmatamento x Economia', fontsize=14)
plt.show()